# Reproducing the results: heat deaths missing from occupational burden estimates

This notebook goes from the five input files in `inputs/` to every number in tables 1–3 and in the text of the manuscript, one equation at a time.
Every step runs for all 21 countries at once; each country is one row, so any single country can be audited by reading its row.

**Input data:** The five datasets in `inputs/` are the raw data files used directly in this analysis. They contain the country- and region-level estimates selected from GBD 2023 and ILOSTAT; the notebook does not perform the original data extraction.

| File | Content |
|---|---|
| `1_GBD_deaths_by_country_men.csv` | GBD deaths attributable to occupational risks and high temperature among men aged 15–64, by country, year, age group, and quantity |
| `2_GBD_overlap_inputs_men.csv` | For causes shared by heat and occupational risks: occupational deaths and the corresponding heat-attributable fraction, by country, year, cause, and age group |
| `3_GBD_occupational_deaths_region_by_sex.csv` | GBD occupational-risk deaths for the NAME region, reported separately for men, women, and both sexes |
| `4_ILO_employment_by_sector_modelled.csv` | ILO modelled employment estimates by country, year, sex, and economic sector (ISIC Rev. 4) |
| `5_ILO_employment_to_population_ratio_men_modelled.csv` | ILO modelled employment-to-population ratios among men aged 15 and older, by country and year |

The GBD input files include the point estimate (`val`) and the 95% uncertainty interval (`lower`, `upper`). The analysis uses the point estimates (`val`).

**To test another input:** replace a file in `inputs/` (same columns) and run the notebook again. The last section compares every result with the published value and lists any difference.

**Periods.** Study values are annual averages over **2019–2023**; **1990** is a single comparison year. ILO series for Sudan and Palestine end in 2022, so their exposure averages 2019–2022.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from scipy.optimize import brentq
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 20); pd.set_option('display.max_rows', 40)

IN, RES = Path('inputs'), Path('results'); RES.mkdir(exist_ok=True)
YEARS = [2019, 2020, 2021, 2022, 2023]          # study period (annual average)
BASE = 1990                                      # comparison year
RR_VALUES = [1.5, 2.0, 3.0]                      # low, central, high relative risk
GULF = ['Bahrain', 'Kuwait', 'Oman', 'Qatar', 'Saudi Arabia', 'United Arab Emirates']
ILO_ARAB_STATES = GULF + ['Iraq', 'Jordan', 'Lebanon', 'Palestine', 'Syrian Arab Republic', 'Yemen']

deaths = pd.read_csv(IN / '1_GBD_deaths_by_country_men.csv')
ovl = pd.read_csv(IN / '2_GBD_overlap_inputs_men.csv')
bysex = pd.read_csv(IN / '3_GBD_occupational_deaths_region_by_sex.csv')
emp = pd.read_csv(IN / '4_ILO_employment_by_sector_modelled.csv')
ratio = pd.read_csv(IN / '5_ILO_employment_to_population_ratio_men_modelled.csv')
COUNTRIES = sorted(deaths.location.unique()); OTHER = [c for c in COUNTRIES if c not in GULF]
GROUPS = {'Gulf states (6)': GULF, 'Other countries (15)': OTHER, 'All 21 countries': COUNTRIES}
print(len(COUNTRIES), 'countries;', deaths.quantity.nunique(), 'GBD quantities')

## Step 1 — Occupational deaths (O) and heat deaths (H)

For each country and year, sum the GBD age groups 15–64:

- **O** = deaths attributable to all occupational risks combined, all causes
- **H** = deaths attributable to high temperature, all causes − deaths attributable to high temperature from exposure to forces of nature (floods and similar single events)

In [ ]:
def total(quantity):
    d = deaths[deaths.quantity == quantity]
    return d.groupby(['location', 'year']).val.sum()

O = total('Occupational risks, all causes')
H_all = total('High temperature, all causes')
H_fon = total('High temperature, exposure to forces of nature').reindex(H_all.index, fill_value=0)
H = H_all - H_fon
step1 = pd.DataFrame({'O': O, 'H, all causes': H_all, 'H, forces of nature': H_fon, 'H': H})
print('Negative heat-death estimates are retained as published; the Libya 2023 value is explained in the manuscript Methods:')
step1[step1.H < 0].round(1)

## Step 2 — Overlap deaths (L): heat deaths GBD already counts as occupational

Heat and occupational risks share two outcomes in GBD: injuries (through occupational injuries) and COPD (through occupational particulate matter, gases, and fumes). GBD treats the two risks as independent, so for each cause and age group:

**overlap = occupational deaths from that cause × share of deaths from that cause attributable to high temperature**

and L is the sum over causes and age groups. Causes for which GBD gives no heat estimate (blank fraction) contribute 0.

In [ ]:
ovl['overlap'] = ovl.val_occupational_deaths * ovl.val_heat_fraction.fillna(0)
L = ovl.groupby(['location', 'year']).overlap.sum()
print('Example: Qatar, 2023, road injuries')
ovl[(ovl.location == 'Qatar') & (ovl.year == 2023) & (ovl.cause == 'Road injuries')][['age', 'val_occupational_deaths', 'val_heat_fraction', 'overlap']].round(4)

## Step 3 — Table 1: averages over 2019–2023, and 1990

Country value = mean of the five annual values. Group rows = sums of country rows.

In [ ]:
yearly = pd.DataFrame({'O': O, 'H': H, 'L': L})
cur = yearly[yearly.index.get_level_values('year').isin(YEARS)].groupby('location').mean()
b90 = yearly.xs(BASE, level='year')

def add_groups(df):
    return pd.concat([df, pd.DataFrame({g: df.loc[m].sum() for g, m in GROUPS.items()}).T])

t1 = pd.concat({'2019-2023 average': add_groups(cur), '1990': add_groups(b90)}, axis=1)
t1[('2019-2023 average', 'L % of O')] = t1[('2019-2023 average', 'L')] / t1[('2019-2023 average', 'O')] * 100
t1 = t1.loc[list(cur.sort_values('H', ascending=False).index) + list(GROUPS)]
t1.to_csv(RES / 'Table1.csv'); t1.round(1)

## Step 4 — Men working outdoors (E), equation 1

**E = employment-to-population ratio (men 15+) × share of employed men working in agriculture, forestry, and fishing (ISIC A), mining and quarrying (B), or construction (F)**

E is calculated for each year and then averaged over 2019–2023. The number of men working outdoors is the sum of employment in A, B, and F, averaged over the same years.

In [ ]:
SECT = ['A. Agriculture, forestry and fishing', 'B. Mining and quarrying', 'F. Construction']
w = emp.pivot_table(index=['location', 'year', 'sex'], columns='sector_ISIC_Rev4', values='employment_thousands')
w['outdoor'] = w[SECT].sum(axis=1)
men = w.xs('Male', level='sex').copy()
men['share_outdoor'] = men.outdoor / men.Total
men = men.join(ratio.set_index(['location', 'year']).employment_to_population_ratio_men_15plus_pct / 100)
men['E'] = men.employment_to_population_ratio_men_15plus_pct * men.share_outdoor
ex = men.groupby('location').agg(E=('E', 'mean'), outdoor_thousands=('outdoor', 'mean'), years=('E', 'size'))
B = cur.join(ex)                                            # one row per country: O, H, L, E
B[['E', 'outdoor_thousands', 'years']].assign(E_pct=B.E * 100).sort_values('E', ascending=False).round(3)

## Step 5 — Additional occupational heat deaths, equations 2–5

2. Share of heat deaths attributable to outdoor work: **C = E (RR − 1) ÷ [1 + E (RR − 1)]** (Levin's attributable fraction, applied to heat-attributable deaths)
3. Heat deaths attributable to outdoor work: **A = H × C**
4. Additional occupational heat deaths: **U = A − L**, set to 0 if negative
5. Addition to the occupational burden: **U ÷ O × 100**

RR = relative risk of heat-attributable death in outdoor workers compared with other men; 1·5, 2 (central), and 3. Group rows: sums of A, L, U, and O over countries.

In [ ]:
def model(E, RR, b=B):
    C = E * (RR - 1) / (1 + E * (RR - 1))
    A = b.H * C
    U = (A - b.L).clip(lower=0)
    return pd.DataFrame({'E': E, 'C': C, 'A': A, 'L': b.L, 'U': U, 'O': b.O, 'addition_pct': U / b.O * 100})

def group_rows(r):
    out = {}
    for g, m in GROUPS.items():
        s = r.loc[m, ['A', 'L', 'U', 'O']].sum(); s['addition_pct'] = s.U / s.O * 100; out[g] = s
    return pd.DataFrame(out).T

R = {rr: model(B.E, rr) for rr in RR_VALUES}
t2 = pd.DataFrame({'Men working outdoors (%)': B.E * 100, 'Men working outdoors (millions)': B.outdoor_thousands / 1000,
                   'Share of heat deaths attributable to outdoor work, RR=2 (%)': R[2].C * 100,
                   'Additional occupational heat deaths per year, RR=2': R[2].U,
                   'Addition RR=1.5 (%)': R[1.5].addition_pct, 'Addition RR=2 (%)': R[2].addition_pct, 'Addition RR=3 (%)': R[3].addition_pct})
t2 = t2.sort_values('Addition RR=2 (%)', ascending=False)
# group rows; group E = men working outdoors / men aged 15+, group C = sum A / sum H
pop15 = B.outdoor_thousands / B.E
g2 = pd.DataFrame({g: {'Men working outdoors (%)': B.loc[m, 'outdoor_thousands'].sum() / pop15[m].sum() * 100,
                       'Men working outdoors (millions)': B.loc[m, 'outdoor_thousands'].sum() / 1000,
                       'Share of heat deaths attributable to outdoor work, RR=2 (%)': R[2].loc[m, 'A'].sum() / B.loc[m, 'H'].sum() * 100,
                       'Additional occupational heat deaths per year, RR=2': group_rows(R[2]).loc[g, 'U'],
                       'Addition RR=1.5 (%)': group_rows(R[1.5]).loc[g, 'addition_pct'],
                       'Addition RR=2 (%)': group_rows(R[2]).loc[g, 'addition_pct'],
                       'Addition RR=3 (%)': group_rows(R[3]).loc[g, 'addition_pct']} for g, m in GROUPS.items()}).T
t2 = pd.concat([t2, g2]); t2.to_csv(RES / 'Table2.csv'); t2.round(2)

## Step 6 — Table 3: smallest relative risk at which additional occupational heat deaths add t% (5% or 10%)

Setting U ÷ O × 100 = t in equations 2–5 and solving for RR gives an exact formula for each country:

- required share **C\* = (L + t·O/100) ÷ H**
- **RR needed = 1 + C\* ÷ [E (1 − C\*)]**
- if C\* ≥ 1, no RR is large enough ("Not possible"); the largest possible addition, reached when all heat deaths are attributed to outdoor work, is **(H − L) ÷ O × 100**

Group rows have no closed form (each country has its own E), so they are solved numerically: the RR at which the group's addition (sum U ÷ sum O) equals t.

In [ ]:
def rr_needed(b, t):
    Cs = (b.L + t * b.O / 100) / b.H
    return (1 + Cs / (b.E * (1 - Cs))).where((Cs > 0) & (Cs < 1))

t3 = pd.DataFrame({'RR for +5%': rr_needed(B, 5), 'RR for +10%': rr_needed(B, 10),
                   'Maximum possible addition (%)': (B.H - B.L).clip(lower=0) / B.O * 100})
t3 = t3.sort_values(['RR for +10%', 'RR for +5%'], na_position='last')
def rr_group(g, t):
    f = lambda rr: group_rows(model(B.E, rr)).loc[g, 'addition_pct'] - t
    return brentq(f, 1.0001, 1e6) if f(1e6) > 0 else np.nan
g3 = pd.DataFrame({f'RR for +{t}%': {g: rr_group(g, t) for g in GROUPS} for t in (5, 10)})
# check: the closed form gives the same addition as equations 2-5
chk = model(B.E.loc[t3['RR for +10%'].dropna().index], t3['RR for +10%'].dropna(), B.loc[t3['RR for +10%'].dropna().index]).addition_pct
print('Closed form check (addition at RR needed should be 10%): max deviation', float((chk - 10).abs().max()))
t3 = pd.concat([t3, g3]); t3.to_csv(RES / 'Table3.csv'); t3.round(3)

## Step 7 — Other numbers quoted in the text

Each line gives the formula; inputs come from steps 1–6 or from the input files.

In [ ]:
c, g = t1['2019-2023 average'], t1['1990']
ALL, GUL, OTH = 'All 21 countries', 'Gulf states (6)', 'Other countries (15)'
x = {}
# Methods
s = bysex[bysex.year.isin(YEARS)].groupby('sex').val.sum()
x["Men's share of occupational-risk deaths, region, 15-64 y (%) = men / both sexes"] = s.Male / s.Both * 100
wf = w[['outdoor']].groupby(['location', 'sex']).mean().outdoor.unstack()      # mean over years, by sex
x['Women share of workers in A, B, F, region (%) = sum women / sum all'] = wf.Female.sum() / wf.Total.sum() * 100
x['Women share of workers in A, B, F, median across countries (%)'] = (wf.Female / wf.Total).median() * 100
# Results
x['Heat deaths in Iraq, Sudan, Egypt as % of all heat deaths'] = c.loc[['Iraq', 'Sudan', 'Egypt'], 'H'].sum() / c.loc[ALL, 'H'] * 100
inj = ovl[(ovl.occupational_risk == 'Occupational injuries') & ovl.year.isin(YEARS)]
x['Road injuries as % of overlap from injuries'] = inj[inj.cause == 'Road injuries'].overlap.sum() / inj.overlap.sum() * 100
x['Change in heat deaths, 1990 to 2019-2023 (%)'] = (c.loc[ALL, 'H'] / g.loc[ALL, 'H'] - 1) * 100
x['Change in occupational deaths, 1990 to 2019-2023 (%)'] = (c.loc[ALL, 'O'] / g.loc[ALL, 'O'] - 1) * 100
u = group_rows(R[2])
x['Additional occupational heat deaths outside the Gulf, % of all additional deaths (RR=2)'] = u.loc[OTH, 'U'] / u.loc[ALL, 'U'] * 100
# Discussion
x['Heat deaths / occupational deaths, Gulf states'] = c.loc[GUL, 'H'] / c.loc[GUL, 'O']
x['Heat deaths / occupational deaths, other countries'] = c.loc[OTH, 'H'] / c.loc[OTH, 'O']
x['Saudi Arabia: change in occupational deaths since 1990 (%)'] = (c.loc['Saudi Arabia', 'O'] / g.loc['Saudi Arabia', 'O'] - 1) * 100
x['Saudi Arabia: change in heat deaths since 1990 (%)'] = (c.loc['Saudi Arabia', 'H'] / g.loc['Saudi Arabia', 'H'] - 1) * 100
cause = lambda q: deaths[(deaths.quantity == q) & deaths.year.isin(YEARS)].groupby('location').val.sum()
cvd, hall, hfon = cause('High temperature, cardiovascular diseases'), cause('High temperature, all causes'), cause('High temperature, exposure to forces of nature').reindex(COUNTRIES, fill_value=0)
for grp in (GUL, OTH):
    m = GROUPS[grp]; x[f'Cardiovascular diseases as % of heat deaths, {grp}'] = cvd[m].sum() / (hall[m].sum() - hfon[m].sum()) * 100
oinj = cause('Occupational injuries, all injuries'); linj = inj.groupby('location').overlap.sum()
for grp, m in [('ILO Arab States, 12 study countries', ILO_ARAB_STATES), (GUL, GULF)]:
    x[f'Occupational-injury deaths also attributable to heat, {grp} (%)'] = linj[m].sum() / oinj[m].sum() * 100
text = pd.Series(x, name='value'); text.to_csv(RES / 'numbers_in_text.csv'); text.round(2).to_frame()

## Step 8 — Comparison with the published values

Each value is formatted as printed in the manuscript (mid-dot decimals) and compared with `published_values.csv`, which the manuscript was built from.

In [ ]:
def num(v, d=1):
    s = f'{float(v):,.{d}f}'.replace(',', ' ').replace('.', '·')
    return s.replace(' ', '') if d == 0 and abs(v) < 10000 else s
integer = lambda v: num(round(v), 0)
def rng(a, b, d=1):
    a, b = round(a, d), round(b, d); return num(a, d) if a == b else f'{num(a, d)}–{num(b, d)}'
irng = lambda v: rng(v.min(), v.max(), 0)
G5 = [k for k in GULF if k != 'Bahrain']; EISB = ['Egypt', 'Iraq', 'Sudan', 'Bahrain']
T2, T3 = t2, t3
mine = {
 'O_all': integer(c.loc[ALL, 'O']), 'H_all': integer(c.loc[ALL, 'H']), 'H_gulf': integer(c.loc[GUL, 'H']), 'O_gulf': integer(c.loc[GUL, 'O']),
 'H_bahrain': integer(c.loc['Bahrain', 'H']), 'O_bahrain': integer(c.loc['Bahrain', 'O']), 'L_all': integer(c.loc[ALL, 'L']),
 'L_pct_all': num(c.loc[ALL, 'L % of O']), 'L_pct_gulf': num(c.loc[GUL, 'L % of O']), 'L_pct_other': num(c.loc[OTH, 'L % of O']),
 'add_all': num(T2.loc[ALL, 'Addition RR=2 (%)']), 'add_all_lo': num(T2.loc[ALL, 'Addition RR=1.5 (%)']), 'add_all_hi': num(T2.loc[ALL, 'Addition RR=3 (%)']),
 'add_gulf': num(T2.loc[GUL, 'Addition RR=2 (%)']), 'add_gulf_lo': num(T2.loc[GUL, 'Addition RR=1.5 (%)']), 'add_gulf_hi': num(T2.loc[GUL, 'Addition RR=3 (%)']),
 'add_gulf5_range': irng(T2.loc[G5, 'Addition RR=2 (%)']), 'add_bahrain': integer(T2.loc['Bahrain', 'Addition RR=2 (%)']),
 'add_eisb_range': irng(T2.loc[EISB, 'Addition RR=2 (%)']),
 'tip_gulf5_range': rng(T3.loc[G5, 'RR for +10%'].min(), T3.loc[G5, 'RR for +10%'].max()),
 'tip_gulf_range': rng(T3.loc[GULF, 'RR for +10%'].min(), T3.loc[GULF, 'RR for +10%'].max()),
 'tip_gulf_group': num(T3.loc[GUL, 'RR for +10%']), 'tip_eisb_range': rng(T3.loc[EISB, 'RR for +10%'].min(), T3.loc[EISB, 'RR for +10%'].max()),
 'tip_all': num(T3.loc[ALL, 'RR for +10%']),
 'men_out_all': num(B.outdoor_thousands.sum() / 1000), 'men_out_gulf': num(B.loc[GULF, 'outdoor_thousands'].sum() / 1000),
 'men_occ_share': num(x["Men's share of occupational-risk deaths, region, 15-64 y (%) = men / both sexes"]),
 'women_share': num(x['Women share of workers in A, B, F, region (%) = sum women / sum all']),
 'women_median': num(x['Women share of workers in A, B, F, median across countries (%)']),
 'ise_share': num(x['Heat deaths in Iraq, Sudan, Egypt as % of all heat deaths']),
 'road_share': num(x['Road injuries as % of overlap from injuries']),
 'heat_change': integer(x['Change in heat deaths, 1990 to 2019-2023 (%)']),
 'occ_change': num(-x['Change in occupational deaths, 1990 to 2019-2023 (%)']),
 'ratio_gulf': num(x['Heat deaths / occupational deaths, Gulf states']), 'ratio_other': num(x['Heat deaths / occupational deaths, other countries']),
 'saudi_occ': num(-x['Saudi Arabia: change in occupational deaths since 1990 (%)']), 'saudi_heat': integer(x['Saudi Arabia: change in heat deaths since 1990 (%)']),
 'cvd_gulf': num(x[f'Cardiovascular diseases as % of heat deaths, {GUL}']), 'cvd_other': num(x[f'Cardiovascular diseases as % of heat deaths, {OTH}']),
 'ilo_arab': num(x['Occupational-injury deaths also attributable to heat, ILO Arab States, 12 study countries (%)']),
 'ilo_gulf': num(x[f'Occupational-injury deaths also attributable to heat, {GUL} (%)']),
}
pub = pd.read_csv('published_values.csv', index_col=0).value.astype(str)
cmp_ = pd.DataFrame({'reproduced': pd.Series(mine), 'published': pub}).dropna()
cmp_['match'] = cmp_.reproduced == cmp_.published
print(f"{cmp_.match.sum()} of {len(cmp_)} numbers in the text match exactly")
cmp_.to_csv(RES / 'comparison_text.csv'); cmp_

In [ ]:
# tables: largest absolute difference against the published tables (rounding in the published files: 1-3 decimals)
def diff(mine, f, **kw):
    p = pd.read_csv(Path('published') / f, **kw)
    a, b = mine.reindex(p.index), p
    a = a.reindex(columns=b.columns) if a.columns.nlevels == b.columns.nlevels else a
    return float(np.nanmax(np.abs(a.values.astype(float) - b.values.astype(float))))
t1p = t1.rename(columns={'O': 'occupational_deaths', 'H': 'heat_deaths', 'L': 'overlap', 'L % of O': 'overlap % of occupational'})
print('Table 1, largest difference:', round(diff(t1p, 'Table1_GBD_quantities.csv', header=[0, 1], index_col=0), 4))
print('Table 2 countries, largest difference:', round(diff(t2.drop(list(GROUPS)), 'Table2_uncounted_heat_deaths.csv', index_col=0), 4))
print('Table 3 countries, largest difference:', round(diff(t3.drop(list(GROUPS)), 'Table3_tipping_point.csv', index_col=0), 4))
print('Table 3 groups, largest difference:', round(diff(t3.loc[list(GROUPS), ['RR for +5%', 'RR for +10%']], 'Table3_groups.csv', index_col=0), 4))
with pd.ExcelWriter(RES / 'results.xlsx') as xw:
    t1.round(2).to_excel(xw, 'Table1'); t2.round(3).to_excel(xw, 'Table2'); t3.round(3).to_excel(xw, 'Table3')
    text.to_frame().to_excel(xw, 'Numbers in text'); cmp_.to_excel(xw, 'Comparison with manuscript')
print('saved: results/Table1-3.csv, numbers_in_text.csv, comparison_text.csv, results.xlsx')